# Exploração dos dados: receita 2025

In [349]:
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format

df = pd.read_csv("../data/raw/receita_2025.csv", sep=",", decimal=".", encoding="utf-8")

df.columns

Index(['ANO', 'MES', 'COD_MUNICIPIO', 'NOME_MUNICIPIO', 'CD_ORGAO',
       'NOME_ORGAO', 'CD_RECEBIMENTO', 'CD_ORGAO_ORCAMENTARIO',
       'NOME_ORGAO_ORCAMENTARIO', 'CD_UNIDADE_ORCAMENTARIA', 'NOME_UNIDADE',
       'TP_UNIDADE', 'CD_CONTA_SG', 'DS_CONTA_SG', 'NIVEL', 'NR_NIVEL',
       'CD_RECURSO', 'NM_RECURSO', 'CD_FONTE_RECURSO', 'CD_ACOMP_EXEC_ORC',
       'CAR_PECULIAR', 'DS_CAR_PECULIAR', 'VL_ORCADO', 'VL_ARRECADADO'],
      dtype='str')

## Visão geral

Anotações sobre a exploração do arquivo receita_2025.csv:
- Ela tem 24 colunas e 672 linhas
- Algumas colunas, como 'NOME_UNIDADE' e 'TP_UNIDADE', têm apenas 452 linhas preenchidas, enquanto outras têm todas as linhas preenchidas
- 'NM_RECURSO' tem zero linhas preenchidas
- A coluna 'NIVEL' tem apenas dados como 'S' ou 'A'. Sendo 452 linhas com A e 220 linhas com S
- A coluna 'NR_NIVEL' tem apenas números de 1 a 9. E a maioria das linhas tem dados entre 7 e 9
- As linhas com 'NIVEL' = A possuem 'NOME_UNIDADE', enquanto as com 'NIVEL' = S têm 'NOME_UNIDADE' vazio

## NIVEL e NR_NIVEL

In [350]:
filtro_nivel_a = df["NIVEL"] == "A"
filtro_nivel_s = df["NIVEL"] == "S"
filtro_nr_nivel_1 = df["NR_NIVEL"] == 1
filtro_nr_nivel_2 = df["NR_NIVEL"] == 2
filtro_nr_nivel_3 = df["NR_NIVEL"] == 3
filtro_nr_nivel_4 = df["NR_NIVEL"] == 4
filtro_nr_nivel_5 = df["NR_NIVEL"] == 5
filtro_nr_nivel_6 = df["NR_NIVEL"] == 6
filtro_nr_nivel_7 = df["NR_NIVEL"] == 7
filtro_nr_nivel_8 = df["NR_NIVEL"] == 8


df["VL_ARRECADADO"].sum()

np.float64(10455427845.77)

O valor arrecadado de todo o dataframe é R$ 10,455,427,845.77. Mais de seis vezes o valor apenas com NIVEL igual A (R$ 1,549,096,934.03), então somar tudo conta o mesmo dinheiro várias vezes.

In [351]:
df[filtro_nr_nivel_1][["DS_CONTA_SG", "CD_CONTA_SG"]]

,DS_CONTA_SG,CD_CONTA_SG
0,Receitas Correntes,100000000000000
559,Receitas de Capital,200000000000000


In [352]:
print(df[filtro_nr_nivel_1]["VL_ARRECADADO"].sum())
print(df[filtro_nr_nivel_2]["VL_ARRECADADO"].sum())
print(df[filtro_nr_nivel_3]["VL_ARRECADADO"].sum())
print(df[filtro_nr_nivel_4]["VL_ARRECADADO"].sum())

1549096934.03
1549096934.03
1549096934.0300004
1549096934.03


Cada NR_NIVEL, do 1 ao 4, soma exatamente o mesmo valor arrecadado total (R$ 1,549,096,934.03, o mesmo das linhas com NIVEL igual A). Isso provavelmente significa que as linhas se repetem e tem a ver com a decomposição da categoria em níveis.

In [353]:
pd.crosstab(df["NR_NIVEL"], df["NIVEL"])

NIVEL,A,S
NR_NIVEL,,
1,0,2
2,0,9
3,0,20
4,0,39
5,0,67
6,0,24
7,56,54
8,278,5
9,118,0


Do NR_NIVEL 1 ao 6 só existem linhas S. A partir do 7, as linhas A e S se misturam.

Conclusão: NR_NIVEL é o nível da hierarquia. NIVEL A é a ponta da hierarquia, e NIVEL S agrupa as linhas de baixo.

### Teste: a linha S é a soma das linhas A abaixo dela

Pego a linha 3 (NIVEL S) e procuro as linhas A cujo CD_CONTA_SG começa com o mesmo prefixo (o código sem os zeros do fim).

In [354]:
df.loc[3]

ANO                                               2025
MES                                                 12
COD_MUNICIPIO                                      144
NOME_MUNICIPIO                                 PELOTAS
CD_ORGAO                                         54400
NOME_ORGAO                               PM DE PELOTAS
CD_RECEBIMENTO                                  374669
CD_ORGAO_ORCAMENTARIO                                0
NOME_ORGAO_ORCAMENTARIO                            NaN
CD_UNIDADE_ORCAMENTARIA                              0
NOME_UNIDADE                                       NaN
TP_UNIDADE                                         NaN
CD_CONTA_SG                            111200000000000
DS_CONTA_SG                Impostos sobre o Patrimônio
NIVEL                                                S
NR_NIVEL                                             4
CD_RECURSO                                           0
NM_RECURSO                                         NaN
CD_FONTE_R

In [355]:
linha = 3
valor_s = df.loc[linha]["VL_ARRECADADO"]
codigo = df.loc[linha]["CD_CONTA_SG"]
prefixo = str(codigo).rstrip("0")

filtro_prefixo = df["CD_CONTA_SG"].astype(str).str.startswith(prefixo)
df_filhas = df[filtro_prefixo & filtro_nivel_a]

print(prefixo)
print(len(df_filhas))
print(df_filhas["VL_ARRECADADO"].sum(), valor_s)

1112
10
229995095.65 229995095.65


A linha 3 (Impostos sobre o Patrimônio, NIVEL S) arrecadou 229,995,095.65. As 10 linhas A cujo código começa com 1112 somam o mesmo valor. A linha S é o total das linhas A abaixo dela.

## A fazer

- Repetir nos outros três arquivos (receita 2024 e despesa 2024 e 2025)
- Comparar DS_FUNCAO da despesa entre 2024 e 2025
- Valores negativos na receita
- Tipo do CD_CONTA_SG
- VL_EMPENHADO, VL_LIQUIDADO e VL_PAGO na despesa

In [356]:
receita_2024 = pd.read_csv("../data/raw/receita_2024.csv", sep=",", decimal=".", encoding="utf-8")
receita_2025 = pd.read_csv("../data/raw/receita_2025.csv", sep=",", decimal=".", encoding="utf-8")
despesa_2024 = pd.read_csv("../data/raw/despesa_2024.csv", sep=",", decimal=".", encoding="utf-8")
despesa_2025 = pd.read_csv("../data/raw/despesa_2025.csv", sep=",", decimal=".", encoding="utf-8")

arquivos = {
    "receita_2024": receita_2024,
    "receita_2025": receita_2025,
    "despesa_2024": despesa_2024,
    "despesa_2025": despesa_2025,
}

for nome, tabela in arquivos.items():
    print(nome, tabela.shape)

receita_2024 (679, 24)
receita_2025 (672, 24)
despesa_2024 (2990, 41)
despesa_2025 (2894, 41)


In [357]:
list(receita_2024.columns) == list(receita_2025.columns)

True

In [358]:
list(despesa_2024.columns) == list(despesa_2025.columns)

True

In [359]:
def testa_s_contra_a(tabela: pd.DataFrame, linha: int) -> None:
    """Confere se uma linha S é a soma das linhas A abaixo dela."""

    if tabela.loc[linha, "NIVEL"] == "S":

        valor_s = tabela.loc[linha]["VL_ARRECADADO"]
        codigo = tabela.loc[linha]["CD_CONTA_SG"]
        prefixo = str(codigo).rstrip("0")
        filtro_nivel_a = tabela["NIVEL"] == "A"

        filtro_prefixo = tabela["CD_CONTA_SG"].astype(str).str.startswith(prefixo)
        df_filhas = tabela[filtro_prefixo & filtro_nivel_a]
        valor_df_filhas = df_filhas["VL_ARRECADADO"].sum()
        print(f"Codigo: {prefixo}")
        print(f"Total de linhas filhas: {len(df_filhas)}")
        print(f"Total de VL_ARRECADADO das linhas filhas: {valor_df_filhas:,.2f}")
        print(f"VL_ARRECADADO da linha S: {valor_s:,.2f}")


        if valor_s.round(2) == valor_df_filhas.round(2):
            print(f"A linha S é a soma das linhas A abaixo dela.!")
        else:
            print("A linha S NÃO é a soma das linhas A abaixo dela!")
    else:
        print("A linha deve ter NIVEL S")

testa_s_contra_a(receita_2025, 22)
# receita_2025[receita_2025["NIVEL"] == "S"].head(20)

Codigo: 1114
Total de linhas filhas: 5
Total de VL_ARRECADADO das linhas filhas: 140,985,949.00
VL_ARRECADADO da linha S: 140,985,949.00
A linha S é a soma das linhas A abaixo dela.!


In [360]:
def testa_s_contra_a(tabela: pd.DataFrame, linha: int) -> None:
    """Confere se uma linha S é a soma das linhas A abaixo dela."""

    if tabela.loc[linha, "NIVEL"] == "S":

        valor_s = tabela.loc[linha]["VL_ARRECADADO"]
        codigo = tabela.loc[linha]["CD_CONTA_SG"]
        prefixo = str(codigo).rstrip("0")
        filtro_nivel_a = tabela["NIVEL"] == "A"

        filtro_prefixo = tabela["CD_CONTA_SG"].astype(str).str.startswith(prefixo)
        df_filhas = tabela[filtro_prefixo & filtro_nivel_a]
        valor_df_filhas = df_filhas["VL_ARRECADADO"].sum()
        print(f"Codigo: {prefixo}")
        print(f"Total de linhas filhas: {len(df_filhas)}")
        print(f"Total de VL_ARRECADADO das linhas filhas: {valor_df_filhas:,.2f}")
        print(f"VL_ARRECADADO da linha S: {valor_s:,.2f}")


        if valor_s.round(2) == valor_df_filhas.round(2):
            print(f"A linha S é a soma das linhas A abaixo dela.!")
        else:
            print("A linha S NÃO é a soma das linhas A abaixo dela!")
    else:
        print("A linha deve ter NIVEL S")

testa_s_contra_a(receita_2025, 22)

Codigo: 1114
Total de linhas filhas: 5
Total de VL_ARRECADADO das linhas filhas: 140,985,949.00
VL_ARRECADADO da linha S: 140,985,949.00
A linha S é a soma das linhas A abaixo dela.!


In [361]:
df_filtrado = df[filtro_nr_nivel_6]
serie_limpa = df_filtrado["CD_CONTA_SG"].astype(str).str.rstrip("0")
serie_limpa.str.len().mean()
df["caracteres_codigo"] = df["CD_CONTA_SG"].astype(str).str.rstrip("0").str.len()
pd.crosstab(df["NR_NIVEL"], df["caracteres_codigo"])

caracteres_codigo,1,2,3,4,5,6,7,8,10,14,15
NR_NIVEL,,,,,,,,,,,
1,2,0,0,0,0,0,0,0,0,0,0
2,0,9,0,0,0,0,0,0,0,0,0
3,0,0,20,0,0,0,0,0,0,0,0
4,0,0,0,39,0,0,0,0,0,0,0
5,0,0,0,0,15,52,0,0,0,0,0
6,0,0,0,0,0,0,24,0,0,0,0
7,0,0,0,0,0,0,0,54,0,0,56
8,0,0,0,0,0,0,0,0,5,0,278
9,0,0,0,0,0,0,0,0,0,5,113


In [362]:
df_filtrado = df[filtro_nivel_a]
df_filtrado.groupby(["NR_NIVEL", "NIVEL", "DS_CONTA_SG", "CD_CONTA_SG"])["VL_ARRECADADO"].sum().sample(20)

NR_NIVEL  NIVEL  DS_CONTA_SG                                                                                                       CD_CONTA_SG    
8         A      Multas Previstas em Legislação Específica - Art. 7º Lei 4.421 - Vigilância Sanitária - Multas e Juros             191101020300001        5,122.25
                 Contrato 78.458 - BRDE                                                                                            211201010400001    3,472,506.17
                 Rede Bem Cuidar e UBS - Qualifica Ações Puericultura - Portaria SES nº 1.228/25                                   242150010700001       50,000.00
                 Incremento MAC - Portaria nº 7.446/2025 - Hospital Beneficência Portuguesa                                        171350213200001    2,000,000.00
                 Alienação de Bens Móveis e Semoventes - Exceto RPPS                                                               221301010200002      193,350.00
                 Fortalecimento do Sis

In [363]:
df["CD_CONTA_SG"] = df["CD_CONTA_SG"].astype(str)
codigo = "111250"
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df_filtrado_codigo = df[(filtro_codigo) & (filtro_nivel_a)]
# df_filtrado_codigo[["NOME_ORGAO_ORCAMENTARIO", "NIVEL", "NR_NIVEL", "DS_CONTA_SG", "CD_CONTA_SG", "VL_ARRECADADO"]].iloc[0]
df_filtrado_codigo.iloc[0]


ANO                                                                     2025
MES                                                                       12
COD_MUNICIPIO                                                            144
NOME_MUNICIPIO                                                       PELOTAS
CD_ORGAO                                                               54400
NOME_ORGAO                                                     PM DE PELOTAS
CD_RECEBIMENTO                                                        374669
CD_ORGAO_ORCAMENTARIO                                                     40
NOME_ORGAO_ORCAMENTARIO                      Secretaria Municipal da Fazenda
CD_UNIDADE_ORCAMENTARIA                                                   44
NOME_UNIDADE                                 Secretaria Municipal da Fazenda
TP_UNIDADE                                                               PRM
CD_CONTA_SG                                                  111250010000001

In [364]:
# Nível 1
codigo_folha = "111250010000001"
codigo = codigo_folha[:1]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_1) & (filtro_codigo)].iloc[0]


ANO                                      2025
MES                                        12
COD_MUNICIPIO                             144
NOME_MUNICIPIO                        PELOTAS
CD_ORGAO                                54400
NOME_ORGAO                      PM DE PELOTAS
CD_RECEBIMENTO                         374669
CD_ORGAO_ORCAMENTARIO                       0
NOME_ORGAO_ORCAMENTARIO                   NaN
CD_UNIDADE_ORCAMENTARIA                     0
NOME_UNIDADE                              NaN
TP_UNIDADE                                NaN
CD_CONTA_SG                   100000000000000
DS_CONTA_SG                Receitas Correntes
NIVEL                                       S
NR_NIVEL                                    1
CD_RECURSO                                  0
NM_RECURSO                                NaN
CD_FONTE_RECURSO                            0
CD_ACOMP_EXEC_ORC                           0
CAR_PECULIAR                                0
DS_CAR_PECULIAR                 Nã

In [365]:
# Nível 2
codigo = codigo_folha[:2]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_2) & (filtro_codigo)].iloc[0]

ANO                                                               2025
MES                                                                 12
COD_MUNICIPIO                                                      144
NOME_MUNICIPIO                                                 PELOTAS
CD_ORGAO                                                         54400
NOME_ORGAO                                               PM DE PELOTAS
CD_RECEBIMENTO                                                  374669
CD_ORGAO_ORCAMENTARIO                                                0
NOME_ORGAO_ORCAMENTARIO                                            NaN
CD_UNIDADE_ORCAMENTARIA                                              0
NOME_UNIDADE                                                       NaN
TP_UNIDADE                                                         NaN
CD_CONTA_SG                                            110000000000000
DS_CONTA_SG                Impostos, Taxas e Contribuições de Melhoria
NIVEL 

In [366]:
# Nível 3
codigo = codigo_folha[:3]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_3) & (filtro_codigo)].iloc[0]

ANO                                   2025
MES                                     12
COD_MUNICIPIO                          144
NOME_MUNICIPIO                     PELOTAS
CD_ORGAO                             54400
NOME_ORGAO                   PM DE PELOTAS
CD_RECEBIMENTO                      374669
CD_ORGAO_ORCAMENTARIO                    0
NOME_ORGAO_ORCAMENTARIO                NaN
CD_UNIDADE_ORCAMENTARIA                  0
NOME_UNIDADE                           NaN
TP_UNIDADE                             NaN
CD_CONTA_SG                111000000000000
DS_CONTA_SG                       Impostos
NIVEL                                    S
NR_NIVEL                                 3
CD_RECURSO                               0
NM_RECURSO                             NaN
CD_FONTE_RECURSO                         0
CD_ACOMP_EXEC_ORC                        0
CAR_PECULIAR                             0
DS_CAR_PECULIAR              Não se Aplica
VL_ORCADO                   375,126,608.98
VL_ARRECADA

In [367]:
# Nível 4
codigo = codigo_folha[:4]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_4) & (filtro_codigo)].iloc[0]

ANO                                               2025
MES                                                 12
COD_MUNICIPIO                                      144
NOME_MUNICIPIO                                 PELOTAS
CD_ORGAO                                         54400
NOME_ORGAO                               PM DE PELOTAS
CD_RECEBIMENTO                                  374669
CD_ORGAO_ORCAMENTARIO                                0
NOME_ORGAO_ORCAMENTARIO                            NaN
CD_UNIDADE_ORCAMENTARIA                              0
NOME_UNIDADE                                       NaN
TP_UNIDADE                                         NaN
CD_CONTA_SG                            111200000000000
DS_CONTA_SG                Impostos sobre o Patrimônio
NIVEL                                                S
NR_NIVEL                                             4
CD_RECURSO                                           0
NM_RECURSO                                         NaN
CD_FONTE_R

In [368]:
# Nível 5
codigo = codigo_folha[:6]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_5) & (filtro_codigo)]

,ANO,MES,COD_MUNICIPIO,NOME_MUNICIPIO,CD_ORGAO,NOME_ORGAO,CD_RECEBIMENTO,CD_ORGAO_ORCAMENTARIO,NOME_ORGAO_ORCAMENTARIO,CD_UNIDADE_ORCAMENTARIA,...,NR_NIVEL,CD_RECURSO,NM_RECURSO,CD_FONTE_RECURSO,CD_ACOMP_EXEC_ORC,CAR_PECULIAR,DS_CAR_PECULIAR,VL_ORCADO,VL_ARRECADADO,caracteres_codigo
4,2025,12,144,PELOTAS,54400,PM DE PELOTAS,374669,0,NaN,0,...,5,0,NaN,0,0,0,Não se Aplica,"164,362,337.22","184,095,028.53",5


In [369]:
# Nível 6
codigo = codigo_folha[:7]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
df[(filtro_nivel_s) & (filtro_nr_nivel_6) & (filtro_codigo)]

,ANO,MES,COD_MUNICIPIO,NOME_MUNICIPIO,CD_ORGAO,NOME_ORGAO,CD_RECEBIMENTO,CD_ORGAO_ORCAMENTARIO,NOME_ORGAO_ORCAMENTARIO,CD_UNIDADE_ORCAMENTARIA,...,NR_NIVEL,CD_RECURSO,NM_RECURSO,CD_FONTE_RECURSO,CD_ACOMP_EXEC_ORC,CAR_PECULIAR,DS_CAR_PECULIAR,VL_ORCADO,VL_ARRECADADO,caracteres_codigo


In [380]:
# Nível a
codigo = codigo_folha[:6]
filtro_codigo = df["CD_CONTA_SG"].str.startswith(codigo)
print(df[(filtro_nivel_a) & (filtro_codigo)][["VL_ARRECADADO", "NIVEL", "NR_NIVEL"]])
print(df[(filtro_nivel_a) & (filtro_codigo)]["VL_ARRECADADO"].sum().round(2))

    VL_ARRECADADO NIVEL  NR_NIVEL
5  148,422,688.20     A         7
6            0.06     A         7
7            0.00     A         7
8    2,575,923.48     A         7
9   22,059,179.86     A         7
10       1,398.00     A         7
11  11,035,838.93     A         7
184095028.53


## O que está comprovado
- A receita é uma hierarquia de categorias e subcategorias. O NR_NIVEL é a profundidade, e o CD_CONTA_SG mostra a ligação: o código do filho começa com o código do pai (o ramo do IPTU, do nível 1 ao 5, comprova).
- NIVEL = S agrupa, NIVEL = A é a ponta. A linha S é a soma das linhas A abaixo dela (confirmado em linhas que você testou, e na verificação completa que fiz: 232 de 238 em 2024 e 216 de 220 em 2025).
- O que cada NIVEL significa. A (analítica) é a ponta da hierarquia: é onde o valor é registrado, e as linhas A só aparecem do NR_NIVEL 7 ao 9. S (sintética) é uma categoria que agrupa as de baixo, e vai do NR_NIVEL 1 ao 8. Exemplo: a linha S do IPTU (nível 5) arrecadou R$ 184.095.028,53, e as 7 linhas A sob ela somam exatamente esse valor.
- Somar a tabela inteira conta o mesmo dinheiro várias vezes. Para o total real, some só NIVEL == "A". Isso vai direto para a Etapa 5.
- O tamanho do prefixo depende do nível (1, 2, 3, 4, 6, 7, 8, 10 caracteres), e o rstrip("0") erra no nível 5, por cortar um zero legítimo.
- A hierarquia pula níveis. O IPTU vai do nível 5 direto para a ponta no nível 7.